# AMADS Coding Notebooks

## GCD measurement

1. Comparison of standard vs musically-informed GCD measurement
(spoiler: musically-informed is better)
2. Visualise GCD across variable window-size grid.

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

## <font color='Green'> GCD measurement

GCD measurement is simple and lossless for integers and fraction.

In [ ]:
from amads.algorithms.gcd import *
fraction_gcd([Fraction(1, 2), Fraction(2, 3), Fraction(5, 12)])

Floats are more complex: they require estimation, and to make estimates effective requires domain knowledge.

In [ ]:
test_case = float_gcd_pair(0.6667, 1)

In [ ]:
round(test_case, 3) # failure

In [ ]:
# Leaving the value the same, but changing the tolerance to accommodate is better:
round(float_gcd_pair(0.6667, atol=0.001, rtol=0.001), 3) # success

In [ ]:
# But this same kind of tolerance adjustment can make errors for other, common musical values.
# 15/16 is a common musical value for which the finer tolerance is effective:
fifteen_sixteenths = 15/16
round(1 / float_gcd_pair(fifteen_sixteenths)) # success

In [ ]:
round(1 / float_gcd_pair(fifteen_sixteenths, atol=0.001, rtol=0.001)) # success

In [ ]:
fifteen_sixteenths_3dp = round(fifteen_sixteenths, 3)
round(1 / float_gcd_pair(fifteen_sixteenths_3dp)) # failure

In [ ]:
round(1 / float_gcd_pair(fifteen_sixteenths_3dp, atol=0.001, rtol=0.001)) # failure

In the simplest case, a source records its metrical positions exactly,
including fractional values as needed.
We provide functionality for standard, general algorithms in these cases
(greatest common denominator and fraction estimation)
which are battle-tested and computationally efficient.

In metrically simple and regular cases like chorales, this value might be
the eighth note, for instance.
In other cases, it gets more complex.
For example, Beethoven's Opus 10 Nr.2 Movement 1 includes
a triplet 16th turn figure in measure 1
(tatum = 1/6 division of the quarter note)
and also dotted rhythms that pair a dotted 16th with a 32nd note from measure 5
(tatum = 1/8 division of the quarter).
So to catch these cases in the first 5 measures, we need the
lowest common multiple of 6 and 8, i.e., 24 per quarter (or 48 bins per 2/4 measure).

In cases of extreme complexity, there may be a “need” for a
considerably shorter tatum pulse (and, equivalently, a greater number of bins).
This is relevant for some modern music, as well as cases where
grace notes are assigned a specific metrical position/duration
(though in many encoded standards, grace notes are not assigned
separate metrical positions).

Moreover, there are musical sources that do not encode fractional time
values, but rather approximation with floats. These include any:

  - frame-wise representations of time (including MIDI and any attempted
    transcription from audio),
  - processing via code libraries that likewise convert fractions to floats,
  - secondary representations like most CSVs.

As division by 3 leads to rounding, approximation, and floating point errors,
and as much music involves those divisions, this is widely relevant.

The standard algorithms often fail in these contexts, largely because symbolic music
tends to prioritise certain metrical divisions over others.
For example, 15/16 is a commonly used metrical position (largely because 16 is a power of 2), but 14/15 is not.
That being the case, while 14/15 might be a better mathematical fit for approximating a value,
it is typically incorrect as the musical solution.
We can use the term “incorrect” advisedly here because
the floats are secondary representations of a known fractional ground truth.
Doctests demonstrate some of these cases.

---

## <font color='Green'> Test a More Musical Solution

In [ ]:
from amads.time.meter.grid import generate_n_smooth_numbers, approximate_pulse_match_with_priority_list

In [ ]:
totes = generate_n_smooth_numbers([2, 3], invert=True)
totes

In [ ]:
# Exact fractions: all should pass
for t in totes:
    assert approximate_pulse_match_with_priority_list(t) == t

### Progressive float degradation: method comparison

Each value in `totes` is an exact `Fraction`. We convert each to a float, then round
to decreasing decimal precision (8 dp down to 1 dp) and test whether each estimator recovers
the original fraction.

Two estimators are compared side-by-side:
- `approximate_pulse_match_with_priority_list` — musically-informed, uses a priority list of smooth denominators
- `float_gcd_pair` — general-purpose mathematical GCD estimator

A *success* is defined as the recovered value equalling the exact original `Fraction`.


In [ ]:
def degradation_table(fractions, estimator, dp_range=range(8, 0, -1)):
    """
    For each decimal-precision level in `dp_range`,
    round every fraction's float representation to that many decimal places and
    test whether `estimator` recovers the original fraction.

    Parameters
    ----------
    fractions : list of Fraction
        Ground-truth values (exact).
    estimator : callable
        Function that takes a float and returns a Fraction (or Fraction-comparable).
        In our case, we compare `approximate_pulse_match_with_priority_list` with `float_gcd_pair`.
    dp_range : iterable of int
        Decimal precisions to test, e.g. range(8, 0, -1).

    Returns
    -------
    list of dict with keys: dp, successes, total, rate, failures
    """
    results = []
    total = len(fractions)
    for dp in dp_range:
        successes = 0
        failures = []
        for exact in fractions:
            degraded = round(float(exact), dp)
            recovered = estimator(degraded)
            if recovered == exact:
                successes += 1
            else:
                failures.append((exact, degraded, recovered))
        results.append({
            'dp': dp,
            'successes': successes,
            'total': total,
            'rate': f'{successes}/{total}',
            'failures': failures,
        })
    return results


In [ ]:
table_approx = degradation_table(totes, approximate_pulse_match_with_priority_list)
table_gcd    = degradation_table(totes, float_gcd_pair)


#### Summary table: both methods side by side

Columns: `dp` | `approx_pulse` successes | `float_gcd` successes | Δ (approx − gcd)


In [ ]:
M = len(totes)
hdr = f"{'dp':>4}  {'approx_pulse':>14}  {'float_gcd':>11}  {'Δ':>4}"
print(hdr)
print('-' * len(hdr))
for ra, rg in zip(table_approx, table_gcd):
    delta = ra['successes'] - rg['successes']
    delta_str = f'+{delta}' if delta > 0 else str(delta)
    print(f"{ra['dp']:>4}  {ra['rate']:>14}  {rg['rate']:>11}  {delta_str:>4}")


#### Per-precision detail: which values fail and why

For each precision level where at least one method fails, show the degraded float,
the exact original, and what each estimator returned.


In [ ]:
for ra, rg in zip(table_approx, table_gcd):
    all_exact = {exact for exact, _, _ in ra['failures']} | {exact for exact, _, _ in rg['failures']}
    if not all_exact:
        continue
    print(f"\n=== {ra['dp']} dp — approx_pulse {ra['rate']}, float_gcd {rg['rate']} ===")
    # index failures by exact value for easy lookup
    approx_fail = {e: r for e, _, r in ra['failures']}
    gcd_fail    = {e: r for e, _, r in rg['failures']}
    for exact in sorted(all_exact):
        degraded = round(float(exact), ra['dp'])
        a_res = approx_fail.get(exact, '✓')
        g_res = gcd_fail.get(exact, '✓')
        print(f"  exact={exact!s:<8}  float={degraded}  approx_pulse={a_res}  float_gcd={g_res}")


---

## <font color='Green'> Windowed GCD

Even if we manage to get exact (or estimate) fractions for note events,
when it comes to calculating the GCD tatum value needed to express all these events,
musical sources can be highly variable and a single value for the whole piece is not so instructive.

In cases like these, we may wish to take a windowed approach,
where we calculate GCD for specific areas.

Moreover, to do this systematically, we might want to consider all:
- window sizes (from 1 tatum up to the length of the whole piece)
- start points (incrementing one step at a time for each window size).

Let's explore this first for some toy data and then a real (albeit short) piece.

In [ ]:
monotonic_indices = [0, 1, 2, 4, 6, 7, 10]

In [ ]:
from amads.time.meter import windowed_grid_gcd
windowed_grid_gcd.windowed_gcd(monotonic_indices, plot=True, exclude_redundant=True)

# c.220 for 7 onsets.

## <font color='purple'> Exercise: Apply to Data

Apply this logic to the successive durations in a score, though still a small one.

- Use your own choice of score, or our suggestion (given by the `url` below)
- Parse the score
    - Hint: you can download the parse the score directly from the URL with `from amads.io.readscore import read_score`).
- Get the note start times,
- Convert to indices,
- Calculate and visualise all GCDs.

In [ ]:
url = "https://github.com/MarkGotham/species/raw/refs/heads/main/1x1/082.mxl"

## <font color='crimson'> Solution

In [ ]:
from amads.io.readscore import set_reader_warning_level
set_reader_warning_level("none")

from amads.time.meter import tatum

indices = tatum.score_to_offsets(url, to_indices=True)
indices


In [ ]:
windowed_grid_gcd.windowed_gcd(indices, plot=True, exclude_redundant=True)

Explain this visualisation.
- Where is the anomalous pair of eighth notes?
- How does this make itself known on the grid?

Ends

---